# Variograma experimental y anisotropía

Este notebook genera un campo sintético con mayor continuidad Este–Oeste y construye variogramas omnidireccionales y direccionales. Los datos son ficticios.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(20261009)
n = 65
coords = rng.uniform(0, 1000, size=(n, 2))
dx = coords[:, None, 0] - coords[None, :, 0]
dy = coords[:, None, 1] - coords[None, :, 1]
d_aniso = np.sqrt((dx/360)**2 + (dy/110)**2)
C = 64*np.exp(-(d_aniso**2)) + np.eye(n)*4
z = rng.multivariate_normal(np.full(n, 180.0), C)
i, j = np.triu_indices(n, k=1)
delta = coords[j] - coords[i]
dist = np.linalg.norm(delta, axis=1)
semi = 0.5*(z[j]-z[i])**2
ang = np.degrees(np.arctan2(delta[:, 1], delta[:, 0])) % 180

In [ ]:
ancho = 80
centros = np.arange(40, 680, ancho)
def diferencia_angular(a, d):
    dif = np.abs(a-d)
    return np.minimum(dif, 180-dif)

def variograma(direccion=None, tolerancia=22.5):
    filas = []
    direccion_ok = np.ones_like(dist, dtype=bool) if direccion is None else diferencia_angular(ang, direccion) <= tolerancia
    for h in centros:
        m = direccion_ok & (dist >= h-ancho/2) & (dist < h+ancho/2)
        if m.sum() >= 5:
            filas.append((dist[m].mean(), semi[m].mean(), m.sum()))
    return pd.DataFrame(filas, columns=['distancia', 'semivarianza', 'n_pares'])

vg_omni = variograma()
vg_ew = variograma(0)
vg_ns = variograma(90)
vg_omni

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)
sc = axes[0].scatter(coords[:,0], coords[:,1], c=z, cmap='inferno', s=55, edgecolor='white')
axes[0].set(xlabel='Este (m)', ylabel='Norte (m)', title='Campo sintético', aspect='equal')
fig.colorbar(sc, ax=axes[0], label='Valor')
for vg, etiqueta, color in [(vg_ew, 'Este–Oeste', '#1D4162'), (vg_ns, 'Norte–Sur', '#B56F3A')]:
    axes[1].plot(vg['distancia'], vg['semivarianza'], 'o-', color=color, label=etiqueta)
axes[1].set(xlabel='Distancia media (m)', ylabel='Semivarianza', title='Tolerancia angular ±22.5°')
axes[1].legend(); axes[1].grid(alpha=.25)
plt.show()

## Para interpretar

La curva Este–Oeste debería mostrar menor semivarianza a distancias intermedias. Cambie el ancho de lag y la tolerancia angular: si la conclusión desaparece ante cambios pequeños, la evidencia direccional es frágil. Revise siempre el número de pares.